<a href="https://colab.research.google.com/github/chaseshallow/Ansci4040FinalClassProject/blob/main/Step01PrepareReferenceClip.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Step 01 · Prepare clips

Probes one calving-pen video and cuts its clips into the group's Shared Drive, `CalvingPenPipelineOutputs/<video name>/`:

- `ReferenceClip.mp4`: the window you set below (a stream copy, a few minutes)
- `DevClip.mp4`: optional short re-encoded piece (set `DevStart`)
- `Probe.json`: video + clip info and the settings used

Rerunning **overwrites** these files. The output folder is shared: one person runs a step at a time.
Edit cell 2 and rerun; don't add cells. Save your run with File > Save a copy in Drive, never back to GitHub.

In [ ]:
# 1. Setup: mount Drive, get the pipeline code from GitHub.
from google.colab import drive
drive.mount("/content/drive")

import os, sys
RepoUrl = "https://github.com/chaseshallow/Ansci4040FinalClassProject.git"
RepoBranch = "main"
RepoDir = "/content/Ansci4040FinalClassProject"

if os.path.isdir(RepoDir):
    !git -C "{RepoDir}" fetch -q origin "{RepoBranch}" && git -C "{RepoDir}" checkout -q "{RepoBranch}" && git -C "{RepoDir}" pull -q --ff-only
else:
    !git clone -q --branch "{RepoBranch}" "{RepoUrl}" "{RepoDir}"
sys.path.insert(0, f"{RepoDir}/CalvingPen")

# Only needed after an update: Runtime > Restart session, then rerun from here.
from CalvingPipeline import PrepareClips
from CalvingPipeline.Paths import RawVideoDir, OutputRoot

In [ ]:
# 2. Settings: the only cell to edit.
VideoPath = RawVideoDir / "2026-09-26 00-00-00 Calving Pen 2.avi"
VideoOutputDir = OutputRoot / VideoPath.stem

ClipStart = "03:35:11"   # HH:MM:SS into the video
ClipEnd = None           # None = to the end of the video
DevStart = None          # None = no dev clip; e.g. "03:50:00" for a short re-encoded piece
DevSeconds = None

assert VideoPath.exists(), VideoPath

In [ ]:
# 3. Run: probe + cut the clip(s) (overwrites earlier output).
Summary = PrepareClips(VideoPath, VideoOutputDir, ClipStart, ClipEnd, DevStart, DevSeconds)
Summary

## About this step

**Tool:** ffmpeg (no model). The clip is a *stream copy*: the original pictures are copied, not re-encoded, so an hour takes minutes.

**What we found on the reference video** (`2026-09-26 00-00-00 Calving Pen 2.avi`, window 03:35:11 to the end):
- The AVI header claims **150 fps**; the real rate is **30 fps** (the recorder writes ~5 index entries per frame). The copied clip reports the correct 30 fps.
- `ReferenceClip.mp4`: 1:02:01, 1920×1080, 857 MB.
- Re-encoding (the old `DevClip`) is slow because of the fake 150 fps; leave `DevStart = None` unless a short re-encoded piece is really needed.

Full write-up, glossary and all models: `CalvingPen/README.md` in the repo.